In [130]:
import yaml
from src import paths
import pandas as pd
import numpy as np
import os


In [131]:
# from dotenv import load_dotenv


# from sqlalchemy import create_engine

# # Load environment variables from .env file
# load_dotenv("../private_data/.env")

# host = os.getenv("HOST")
# db = os.getenv("DB")
# port = os.getenv("PORT")
# role = os.getenv("ROLE")
# pw = os.getenv("PASSWORD")
# engine = create_engine(f"postgresql+psycopg2://{role}:{pw}@{host}:{port}/{db}")

# Barter deals

## Load data

In [132]:
df = pd.read_parquet(paths.RAW_DATA_DIR / 'BARTER_DEALS.parquet')
df = df.rename(columns = {'title': 'deal_title', 'description': 'deal_text'})


# Strip timezone from both columns
df['created_at'] = pd.to_datetime(df['created_at']).dt.tz_localize(None)
df['deleted_at'] = pd.to_datetime(df['deleted_at']).dt.tz_localize(None)
df['updated_at'] = pd.to_datetime(df['updated_at']).dt.tz_localize(None)

df.loc[:,'diff_created_deleted'] = df.deleted_at - df.created_at
df.loc[:,'diff_updated_deleted'] = df.updated_at - df.created_at



## Filter dirty rows

- Delete deals that were online for less than $n$ days

In [133]:
print(len(df))
df = df[~(df.diff_created_deleted < pd.Timedelta(days=7))]
print(len(df))

7277
7092


- Remove deals that are too short

In [134]:
print(len(df))
df['text_word_count'] = df['deal_text'].str.split().str.len().fillna(0)
df['title_word_count'] = df['deal_title'].str.split().str.len().fillna(0)

mask_short_text = ((df.title_word_count <= 2) & (df.text_word_count <= 5) & (df.applicants_applications_count < 10))
test = df.loc[mask_short_text,:]
df = df.loc[~mask_short_text,:]
print(len(df))

7092
7049


- remove 'duplicate' deals

In [135]:
print(len(df))
df = df[~(df['deal_title'].str.contains('duplicate') & (df.applicants_applications_count < 5))]
print(len(df))

7049
6880


In [113]:

# # mask_test = (df['deal_title'].str.contains('test') | df['deal_text'].str.contains('test') | df['creators_requirement'].str.contains('test')) & (df['text_word_count'] < 10)
# mask_test = (df['deal_title'].str.contains('test') | df['deal_text'].str.contains('test') | df['creators_requirement'].str.contains('test'))

# a = df[mask_test]
# a

Remove 'test' deals (this is too aggressive, removes valid deals as well)

In [114]:
# print(f"Number of rows before cleaning: {len(df)}")
# mask_test = df['deal_title'].str.contains('test') | df['deal_text'].str.contains('test') | df['creators_requirement'].str.contains('test')
# df_containing_test = df[mask_test]
# df = df[~mask_test]

# print(f"Number of rows after cleaning: {len(df)}")

## Features

- Deal language

In [115]:
from langdetect import detect, detect_langs

deal_langs = []
for i, deal in enumerate(df['deal_text']):
    try:
        lang = detect(deal)
        deal_langs.append(lang)
    except Exception as e:
        print(e)
        deal_langs.append(np.nan)

df['deal_language'] = deal_langs

# Afrikaans is actually Dutch
df.loc[df.deal_language == 'af','deal_language'] = 'nl'

# Filter languages that are not Dutch, English or German (they are gibberish)
allowed_languages = ['nl', 'en', 'de']
test = df[~df['deal_language'].isin(allowed_languages)]

df = df[df['deal_language'].isin(allowed_languages)]
len(df)

No features in text.
No features in text.
No features in text.
No features in text.
No features in text.
No features in text.
No features in text.


6977

In [116]:
test

,applicants_applications_count,content_types,deal_id,main_image,min_social_media_followers,deal_tags,live_since,first_application_at,last_application_at,company_locations,...,tags,gender,featured_image,company_id,partner_id,diff_created_deleted,diff_updated_deleted,text_word_count,title_word_count,deal_language
14,0,"[{'id': 41, 'name': 'Activities', 'slug': 'per...",01985615-068a-00c8-502e-689ad4f46f4f,uploads/deals/01985615-0778-ffff-5f39-475ebacc...,1500,None,NaT,NaT,NaT,None,...,None,None,None,None,01992995-27fe-0065-4f9e-db8ec1e66a27,NaT,0 days 00:00:10.363801,3,4,ro
62,0,"[{'id': 41, 'name': 'Activities', 'slug': 'per...",019bbde4-2b48-00c8-546e-42ac67c258b1,uploads/deals/019bbde3-a9bc-ffff-c07f-0ff60670...,1500,None,2026-01-14 19:03:25.899725,NaT,NaT,None,...,None,unisex,uploads/deals/featured/019bbde4-011e-ffff-fe69...,019bb121-769d-015e-8002-d96220de5cb6,01992995-bd1e-0065-b217-63376c4ac212,NaT,0 days 00:00:00.130933,1,8,ca
92,0,"[{'id': 29, 'name': 'UGC', 'slug': 'Film-Strip...",019bbda5-a0f0-00c8-a531-c652ed176bc8,uploads/deals/019bbda4-f421-ffff-e777-f1737c7a...,1500,None,2026-01-14 17:55:07.245227,NaT,NaT,None,...,None,unisex,uploads/deals/featured/019bbda5-5a32-ffff-c2d5...,019bb121-769d-015e-8002-d96220de5cb6,01992995-bd1e-0065-b217-63376c4ac212,NaT,0 days 00:00:00.124804,1,6,da
111,0,"[{'id': 29, 'name': 'UGC', 'slug': 'Film-Strip'}]",019bbddd-6ee6-00c8-6e0e-23f0c82d9e80,uploads/deals/019bbddb-4fc1-ffff-201a-179cb4f0...,1500,None,2026-01-14 18:56:04.487683,NaT,NaT,None,...,None,unisex,uploads/deals/featured/019bbddb-6ae0-ffff-afa3...,019bb121-769d-015e-8002-d96220de5cb6,01992995-bd1e-0065-b217-63376c4ac212,NaT,0 days 00:00:00.160702,1,11,ca
213,21,"[{'id': 29, 'name': 'UGC', 'slug': 'Film-Strip...",0199c307-9aa0-00c8-e5e0-972ba9903414,uploads/deals/0199c2f3-2deb-ffff-06f8-d0790814...,1500,None,2025-10-08 08:54:39.628585,2025-10-08 10:25:54.216193,2026-01-26 10:36:45.507933,None,...,None,None,None,None,01992995-6531-0065-8ef3-5fd6e1285404,NaT,0 days 21:58:44.699841,6,8,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6726,0,"[{'id': 20, 'name': 'Beauty', 'slug': 'Heart'}...",019c57e7-0650-00c8-a60e-680036aa2e06,uploads/deals/019c57e6-b3f3-ffff-c9e2-2aecaf84...,5000,None,2026-02-13 16:48:04.378345,NaT,NaT,None,...,None,unisex,uploads/deals/featured/019c57e6-cd54-ffff-c836...,019bb11e-1638-015e-6e4d-ac6e8c9bd984,01992995-27fe-0065-4f9e-db8ec1e66a27,NaT,0 days 00:00:00.206479,2,8,ca
6824,23,"[{'id': 41, 'name': 'Activities', 'slug': 'per...",019689e5-d96e-00c8-15fe-ca19e55907a8,uploads/deals/019689e5-d9d6-ffff-701a-3fd92523...,50000,None,2023-11-23 16:07:42.189877,2023-11-24 10:33:55.535940,2024-08-14 07:39:05.886292,"[{""id"": 194, ""name"": ""Barter Agency"", ""town"": ...",...,None,None,None,None,01992995-27fe-0065-4f9e-db8ec1e66a27,NaT,524 days 11:23:22.240045,1,4,et
6886,48,"[{'id': 21, 'name': 'Fashion', 'slug': 'Dress'}]",019ba7ae-83d5-00c8-5102-019af7d7906f,uploads/deals/019ba7ae-4da3-ffff-55a4-3b2a23b9...,5000,None,2026-01-10 11:33:10.886549,2026-01-10 11:42:43.621026,2026-01-29 18:14:38.431801,None,...,None,None,None,None,019b8e66-6edd-0065-4cdc-2e3f5f788eef,NaT,0 days 00:00:00.145291,5,7,so
6943,37,"[{'id': 41, 'name': 'Activities', 'slug': 'per...",019bd6e9-e6fb-00c8-715b-3cb7fa614a43,uploads/deals/019bd6e9-e7d8-ffff-e602-0ee3b0ee...,2500,None,2026-01-19 15:41:32.413631,2026-01-19 15:46:17.679668,2026-01-29 15:53:29.292175,None,...,None,unisex,uploads/deals/featured/019bd6e9-e9b5-ffff-f419...,019bb7ca-e68c-015e-26f3-13bae8378d10,01992995-27fe-0065-4f9e-db8ec1e66a27,NaT,0 days 00:01:20.514630,1,9,fr


## Days since start

In [16]:
df['created_at']

0      2025-02-21 09:47:19.029007
1      2025-03-26 08:56:24.776549
2      2025-02-21 12:14:54.016826
3      2025-03-26 10:58:10.050719
4      2025-03-26 11:14:23.339791
                  ...            
7268   2026-01-25 16:22:15.938891
7269   2026-01-25 16:31:50.902867
7270   2026-01-16 15:47:39.999471
7272   2025-09-15 09:16:48.051838
7273   2026-01-26 08:54:34.812940
Name: created_at, Length: 6472, dtype: datetime64[ns]

In [136]:
# 1. Ensure datetime format
df['created_at'] = pd.to_datetime(df['created_at'])

# 2. Extract a numeric 'Time' feature for correlation (e.g., Days since start)
# We subtract the minimum date to get a "Day 0", "Day 1"... counter
start_date = df['created_at'].min()
df['days_since_start'] = (
    df['created_at'] - start_date).dt.days

- Remove deals that never went live

In [139]:
df = df[~df['go_live_at'].isna()]

In [140]:
df

,applicants_applications_count,content_types,deal_id,main_image,min_social_media_followers,deal_tags,live_since,first_application_at,last_application_at,company_locations,...,tags,gender,featured_image,company_id,partner_id,diff_created_deleted,diff_updated_deleted,text_word_count,title_word_count,days_since_start
0,7,"[{'id': 38, 'name': 'Music', 'slug': 'Music-No...",019689e3-09ed-00c8-ada6-052b1041b584,uploads/deals/019689e3-0a14-ffff-1130-d620d8c2...,2500,None,2023-09-21 07:40:56.211091,2023-09-24 21:36:08.063257,2023-10-05 17:01:53.664059,"[{""id"": ""01KERHSC8805FW4D6D5GG90JMF"", ""name"": ...",...,None,None,None,019bb11c-b0d3-015e-baaf-b9fa5b347794,01992995-1769-0065-49aa-6a67b7f7af7d,NaT,843 days 23:58:36.025130,10,5,13
1,0,"[{'id': 23, 'name': 'Food', 'slug': 'Pizza'}]",019bbd3e-f64e-00c8-0e08-32e34caaaef2,uploads/deals/019bbd3e-75fd-ffff-9bca-d9682beb...,2500,None,2026-01-14 16:02:58.879237,NaT,NaT,"[{""id"": ""01KERHSGAG05FMYGV8JYE2PEAT"", ""name"": ...",...,None,unisex,uploads/deals/featured/019bbd3e-97e0-ffff-3597...,019bb11c-c121-015e-13c7-cd9c8b1c88e4,0199714c-8971-0065-43d5-fca6c121ee7b,NaT,0 days 00:00:00.114981,1,7,860
2,56,"[{'id': 29, 'name': 'UGC', 'slug': 'Film-Strip...",019a118c-fcd5-00c8-6a5c-2b771b7f18a0,uploads/deals/019a118c-a436-ffff-eaa7-e9db6105...,2500,None,2025-10-23 18:25:27.486337,2025-10-23 18:46:51.044149,2025-12-17 17:00:51.616012,None,...,None,None,None,None,01992996-8f9a-0065-30a1-aa9519cd8e27,NaT,0 days 03:34:43.689299,21,6,777
3,0,"[{'id': 21, 'name': 'Fashion', 'slug': 'Dress'}]",019689e5-f489-00c8-8c26-06b3ea0961a5,uploads/deals/019689e5-f54b-ffff-dbed-637e9d2a...,5000,None,2023-11-28 14:59:51.626780,NaT,NaT,None,...,None,None,None,None,01992995-2f32-0065-d8da-ac763ad1f309,NaT,519 days 12:31:19.973315,27,2,82
4,32,"[{'id': 23, 'name': 'Food', 'slug': 'Pizza'}, ...",019689e3-293c-00c8-c1df-735b603c5363,uploads/deals/019689e3-2aac-ffff-196b-7e667fbc...,10000,None,2024-05-22 12:00:33.784921,2024-05-22 12:08:26.158210,2024-06-28 09:56:33.026989,"[{""id"": ""01KERHVBR505FMVR4FSDTRR6AQ"", ""name"": ...",...,None,None,None,019bb11d-aee7-015e-7a46-e31c2aefb012,01992995-7e65-0065-edfb-434b238ddad4,NaT,599 days 19:38:59.828243,35,11,258
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7272,23,"[{'id': 23, 'name': 'Food', 'slug': 'Pizza'}, ...",019a8e95-765c-00c8-3c2d-c82d439d6798,uploads/deals/019a8e8d-91f5-ffff-08c8-be10fa47...,1500,None,2025-11-16 21:32:31.363091,2025-11-16 22:52:14.069908,2026-01-21 06:00:18.685408,"[{""id"": ""01KERJ12X905FNHCM2X7CBZMR3"", ""name"": ...",...,None,None,None,019bb120-8b88-015e-c255-5acb9ca19b34,01992996-00e9-0065-81ca-dbae66b7d097,NaT,56 days 10:07:02.679972,63,4,801
7273,6,"[{'id': 35, 'name': 'Mom', 'slug': 'Baby'}]",019bc149-164d-00c8-3fd0-4ba01f6a2d70,uploads/deals/019bc148-d932-ffff-2c30-8f1668c6...,1500,None,2026-01-20 13:14:26.151491,2026-01-20 23:53:40.243580,2026-01-27 22:06:39.486637,None,...,None,unisex,uploads/deals/featured/019bc149-077a-ffff-5a8b...,019bb7c5-325c-015e-5c9a-3447bec38f70,019b2165-c3cb-0065-2ac7-05958d3ba823,NaT,5 days 02:22:19.044821,9,6,861
7274,41,"[{'id': 20, 'name': 'Beauty', 'slug': 'Heart'}...",019adef8-be38-00c8-79d3-7661c386cf03,uploads/deals/019adef8-a53d-ffff-95a6-fb7e39a9...,5000,None,2025-12-02 12:10:35.116208,2025-12-02 13:29:52.214366,2026-01-14 13:22:40.964880,None,...,None,None,None,None,019ad919-4b0a-0065-06f7-821b102edea9,NaT,0 days 00:03:03.731352,100,7,817
7275,7,"[{'id': 41, 'name': 'Activities', 'slug': 'per...",019bbcee-7663-00c8-dc81-a59920160cfb,uploads/deals/019bbd1a-f01c-ffff-7bb5-d438a57a...,2500,None,2026-01-14 15:28:22.254708,2026-01-15 09:49:58.259999,2026-01-22 10:51:52.051144,None,...,None,unisex,uploads/deals/featured/019bbd1a-bc64-ffff-45e0...,019bbc6a-85c8-015e-6a64-76b39f2df545,019bbc55-4368-0065-222c-b7a23f0b8eb2,NaT,17 days 08:28:55.137698,1,4,860


## Save

In [19]:
processed_data_path = paths.PROCESSED_DATA_DIR / f'{analysis_name}_CLEAN.parquet'
df.to_parquet(processed_data_path)